# 03: Model Training

This notebook trains a fraud detection model using the processed features.

## Setup

In [0]:
%pip install catboost shap imbalanced-learn mlflow

import os, sys, json, pickle
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless rendering on Databricks
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.catboost
import shap

# Add src to path
sys.path.append('/Workspace/Users/mohamed.c.elshenity@gmail.com/fraud/src/model')

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

from train      import train_model
from evaluate   import evaluate_model
from threshold  import find_best_threshold
from shap_explainer import compute_shap_reasons


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.7/96.7 MB 228.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 190.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 127.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 137.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 179.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 151.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 218.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 MB 184.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 12

In [0]:
spark = SparkSession.builder \
    .appName("FraudDetectionModelTraining") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

print(f"Spark version: {spark.version}")


Spark version: 4.1.0


## Load Processed Data

In [0]:
# Load from Delta table written by notebook 02
df = spark.table("fraud_features")
print(f"Processed data loaded: {df.count():,} records")
df.printSchema()


Processed data loaded: 342,566 records
root
 |-- ssn: string (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- acct_num: string (nullable = true)
 |-- profile: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- trans_date: string (nullable = true)
 |-- trans_time: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merchant: string (nullable = true)
 |-- merch_lat: double (nullable = true)
 |--

## Inspect Class Imbalance 

In [0]:
label_dist = df.groupBy("is_fraud").count().toPandas()
total = label_dist["count"].sum()
label_dist["pct"] = (label_dist["count"] / total * 100).round(2)
print("Class distribution:")
print(label_dist.to_string(index=False))

fraud_count   = int(label_dist.loc[label_dist.is_fraud == 1, "count"])
legit_count   = int(label_dist.loc[label_dist.is_fraud == 0, "count"])
imbalance_ratio = round(legit_count / fraud_count, 1)
print(f"\nImbalance ratio (legit:fraud) = {imbalance_ratio}:1")


Class distribution:
 is_fraud  count  pct
        0 334678 97.7
        1   7888  2.3

Imbalance ratio (legit:fraud) = 42.4:1


## Train-Test Split (Time-Based)

In [0]:
# Sort by unix_time (temporal ordering – prevents data leakage)
df = df.orderBy("unix_time")

total_count = df.count()
train_end   = int(total_count * 0.70)
val_end     = int(total_count * 0.85)

window = Window.orderBy("unix_time")
df = df.withColumn("row_idx", F.row_number().over(window))

train_df = df.filter(F.col("row_idx") <= train_end)
val_df   = df.filter((F.col("row_idx") > train_end) & (F.col("row_idx") <= val_end))
test_df  = df.filter(F.col("row_idx") > val_end)

print(f"Train : {train_df.count():,}")
print(f"Val   : {val_df.count():,}")
print(f"Test  : {test_df.count():,}")


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train : 239,796
Val   : 51,385
Test  : 51,385


## Prepare Features and Labels

In [0]:
# All features that come out of the feature-engineering notebooks
FEATURE_COLS = [
    # ── Numeric / Static ──────────────────────────────────────────
    "amt",                    # transaction amount
    "city_pop",               # customer city population
    "hour",                   # hour of day (0-23)
    "day_of_week",            # day of week (1=Sunday)
    "month",                  # calendar month
    # ── Geospatial ────────────────────────────────────────────────
    "distance",               # Euclidean customer-merchant distance
    "haversine_distance",     # Great-circle distance (km)  ← was unused
    # ── Window / Velocity features ────────────────────────────────
    "txn_count_1h",           # # txns by card in last 1 hour
    "txn_count_24h",          # # txns by card in last 24 hours
    "avg_amt_24h",            # avg spend in last 24 hours
    "spend_24h",              # total spend in last 24 hours
    "unique_merchants_24h",   # distinct merchants in last 24 hours
    "time_since_last_txn",    # seconds since previous txn on this card
    # ── Lookup / Fraud-Rate features ──────────────────────────────
    "category_fraud_rate",    # historical fraud rate per category
    "category_txn_count",     # # txns per category (volume signal)
    "merchant_fraud_rate",    # historical fraud rate per merchant
    "merchant_txn_count",     # # txns per merchant
    "state_fraud_rate",       # historical fraud rate per state
    # ── Categorical (passed as-is to CatBoost) ────────────────────
    "category", "merchant", "state", "gender", "city", "zip", "job",
]

# Keep only columns that actually exist in this run's processed table
feature_cols = [c for c in FEATURE_COLS if c in df.columns]
label_col    = "is_fraud"

print(f"Using {len(feature_cols)} features:")
for c in feature_cols:
    print(f"  {c}")


In [0]:
# Convert to pandas for model training (PANDAS BOUNDARY)
X_train = train_df.select(feature_cols).toPandas()
y_train = train_df.select(label_col).toPandas()[label_col]

X_val = val_df.select(feature_cols).toPandas()
y_val = val_df.select(label_col).toPandas()[label_col]

X_test = test_df.select(feature_cols).toPandas()
y_test = test_df.select(label_col).toPandas()[label_col]

print("Data converted to pandas")

## Train Model with MLflow

In [0]:
# Train model
model = train_model(X_train, y_train, X_val, y_val, None)

# Find optimal threshold
best_threshold = find_best_threshold(model, X_val, y_val, n_folds=3)

# Evaluate on test set
metrics = evaluate_model(model, X_test, y_test, threshold=best_threshold)

print(f"Model trained!")
print(f"Test AUC: {metrics['auc']:.4f}")
print(f"Test F1: {metrics['f1']:.4f}")
print(f"Optimal threshold: {best_threshold:.4f}")

## Stop Spark Session

In [0]:
spark.stop()